In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
df_customer = spark.read.option("header","True").option("inferschema","True").csv("/Volumes/shopkart/bronze/raw_files/olist/olist_customers_dataset.csv")\
 .select("*", "_metadata.file_path")


In [0]:
df_customer_new = df_customer.withColumn("_ingested_at",F.current_timestamp())

In [0]:
df_customer_new.show()

In [0]:
df_customer_new.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.customers")

In [0]:
df_order = spark.read.option("header","True").option("inferschema","true").csv("/Volumes/shopkart/bronze/raw_files/olist/olist_orders_dataset.csv")\
.select("*", "_metadata.file_path")

In [0]:
df_order_new=df_order.withColumn("_ingested_at",F.current_timestamp())

In [0]:
df_order_new.schema

In [0]:
my_schema = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', TimestampType(), True), StructField('order_approved_at', TimestampType(), True), StructField('order_delivered_carrier_date', TimestampType(), True), StructField('order_delivered_customer_date', TimestampType(), True), StructField('order_estimated_delivery_date', TimestampType(), True), StructField('file_path', StringType(), False), StructField('_ingested_at', TimestampType(), False)])

df_order_new = spark.read.option("header","True").schema(my_schema).csv("/Volumes/shopkart/bronze/raw_files/olist/olist_orders_dataset.csv")



In [0]:
df_order_new.write.format("delta").mode("overwrite").option("header","true").saveAsTable("shopkart.bronze.orders")

In [0]:
df_order_item = spark.read.option("header","True").option("inferschema","true").csv("/Volumes/shopkart/bronze/raw_files/olist/olist_order_items_dataset.csv")

In [0]:
df_order_item.write.format("delta").mode("overwrite")\
                    .saveAsTable("shopkart.bronze.order_item")

In [0]:
df_product = spark.read.option("header","true").option("inferschema","true").csv("/Volumes/shopkart/bronze/raw_files/olist/olist_products_dataset.csv")

In [0]:
df_product.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.product")

In [0]:
df_payment = spark.read.option("header","True").option("inferschema","True").csv("/Volumes/shopkart/bronze/raw_files/olist/olist_order_payments_dataset.csv")

In [0]:
df_payment.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.payment")

In [0]:
df_seller = spark.read.option("header","true").option("inferschema","True").csv("/Volumes/shopkart/bronze/raw_files/olist/olist_sellers_dataset.csv")

In [0]:
df_seller.write.format("delta").mode("overwrite").saveAsTable("shopkart.bronze.seller")

In [0]:
 files = dbutils.fs.ls("/Volumes/shopkart/bronze/raw_files/olist/")

In [0]:
spark.sql("DROP TABLE IF EXISTS shopkart.bronze._dataset")

In [0]:
import pyspark.sql.functions as F
files = dbutils.fs.ls("/Volumes/shopkart/bronze/raw_files/olist/")

from pyspark.sql.types import StructType, StructField, StringType, IntegerType,TimestampType


schema_1 = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', TimestampType(), True), StructField('order_approved_at', TimestampType(), True), StructField('order_delivered_carrier_date', TimestampType(), True), StructField('order_delivered_customer_date', TimestampType(), True), StructField('order_estimated_delivery_date', TimestampType(), True)])

def reuseable(files):
    for file in files:
        if file.name.endswith("csv"):
            if file.name == "olist_orders_dataset.csv":
                df = spark.read.format("csv").option("header",True).schema(schema_1).load(file.path)
                df = df.withColumns({
                            "_ingested_at": F.current_timestamp(),
                                "_source_file":F.lit(file.path)})
                result = file.name.removeprefix("olist_").replace('_dataset','').removesuffix(".csv")
                df.write.mode("overwrite").saveAsTable(f"shopkart.bronze.{result}")
            else:
                df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(file.path)
                df = df.withColumns({
                            "_ingested_at": F.current_timestamp(),
                                "_source_file":F.lit(file.path)})
                result = file.name.removesuffix('.csv').removesuffix('_dataset').removeprefix('olist_')
                df.write.mode("overwrite").saveAsTable(f"shopkart.bronze.{result}")
                



reuseable(files)